# ICU Charting and Staff - Bronze Layer

This notebook ingests the assigned MIMIC-III source files from S3 and stores them as Bronze Delta tables.

The Bronze layer keeps the source data unchanged apart from adding basic ingestion metadata.

In [0]:
from pyspark.sql import functions as F

# Pipeline configuration
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SOURCE_PATH = "s3://mimic-iii-demo-bucket/"

SOURCE_FILES = {
    "chartevents_raw": "CHARTEVENTS.csv",
    "d_items_raw": "D_ITEMS.csv",
    "datetimeevents_raw": "DATETIMEEVENTS.csv",
    "caregivers_raw": "CAREGIVERS.csv",
    "callout_raw": "CALLOUT.csv"
}

def load_source(file_name):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{SOURCE_PATH}{file_name}")
    )

source_data = {
    table_name: load_source(file_name)
    for table_name, file_name in SOURCE_FILES.items()
}

for table_name, df in source_data.items():
    print(f"{table_name}: {df.count():,} rows")

## Create Bronze Tables

The source files are written to Unity Catalog as Delta tables with the original fields preserved.

Basic ingestion metadata is added so each Bronze record can be traced back to its source file and load time.

In [0]:
# Create Bronze schema
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

# Write source data as Bronze Delta tables
for table_name, df in source_data.items():
    source_file = SOURCE_FILES[table_name]

    bronze_df = (
        df
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"Created: {CATALOG}.{BRONZE_SCHEMA}.{table_name}")

## Bronze Validation

The Bronze tables are checked to confirm that they were created successfully and that the row counts still match the original source files.

In [0]:
# Validate Bronze tables
for table_name in SOURCE_FILES.keys():
    table_path = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    bronze_count = spark.table(table_path).count()
    source_count = source_data[table_name].count()

    print(
        f"{table_name}: "
        f"source={source_count:,} | bronze={bronze_count:,} | "
        f"match={source_count == bronze_count}"
    )

### Bronze Layer Result

All five source datasets were successfully written to the Bronze layer as Delta tables.

The row counts match the original source files, confirming that the ingestion process has preserved the source data correctly. The Bronze tables are now ready to be used as the input for the Silver layer.